In [1]:
import os
import pandas as pd
import numpy as np
import json

try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb

print(f'XGBoost version: {xgb.__version__}')

XGBoost version: 2.1.4


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


#### Constants

In [2]:
str_bucket = 'assessment-alt'
str_task = '05_model'
str_target = 'flt_log_price'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

Bucket: assessment-alt
Task: 05_model


#### Load Data

In [3]:
%%time
X_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_test.csv')
y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv')
y_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_valid.csv')
y_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_test.csv')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'\nFeatures:')
for a, col in enumerate(X_train.columns):
    print(f'{a+1}. {col}')
print()

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


X_train: (62407, 13), y_train: (62407, 1)
X_valid: (21535, 13), y_valid: (21535, 1)
X_test:  (16058, 13), y_test:  (16058, 1)

Features:
1. flt_grade
2. int_is_psa
3. int_is_gem_mint
4. int_card_txn_count
5. flt_card_median_price
6. flt_card_mean_price
7. int_subject_txn_count
8. flt_subject_median_price
9. int_brand_txn_count
10. flt_brand_median_price
11. int_year_sold
12. int_month_sold
13. int_days_since_first_txn

CPU times: user 461 ms, sys: 73.1 ms, total: 534 ms
Wall time: 1.41 s


#### Create DMatrices

In [4]:
dtrain = xgb.DMatrix(X_train, label=y_train[str_target])
dvalid = xgb.DMatrix(X_valid, label=y_valid[str_target])
dtest = xgb.DMatrix(X_test, label=y_test[str_target])
print(f'DMatrices created: train={dtrain.num_row()}, valid={dvalid.num_row()}, test={dtest.num_row()}')

DMatrices created: train=62407, valid=21535, test=16058


#### Define Parameters

For the sake of time and simplicity, we are not performing hyperparameter tuning (e.g., grid search, Bayesian optimization) or recursive feature elimination (RFE) in this iteration. The parameters below are reasonable defaults for XGBoost regression. In a production setting, both hyperparameter tuning and RFE would be worthwhile next steps — tuning to optimize the bias-variance tradeoff, and RFE to identify the minimal feature set that maximizes generalization.

Additionally, we are retaining the card-level price features (flt_card_median_price, flt_card_mean_price) despite their contribution to rapid overfitting. These features encode the same signal as the non-ML exact-match baseline (historical price of the same card) and are legitimately useful for seen cards. Removing them would degrade seen-card predictions without meaningfully improving unseen-card performance. A better approach for future work would be an ensemble strategy or a seen/unseen indicator feature.

In [5]:
dict_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_lambda': 1.0,
    'reg_alpha': 0.1,
    'seed': 42,
    'verbosity': 0,
}
print('Parameters:')
for str_key, val in dict_params.items():
    print(f'  {str_key}: {val}')

Parameters:
  objective: reg:squarederror
  eval_metric: rmse
  learning_rate: 0.05
  max_depth: 6
  min_child_weight: 30
  subsample: 0.8
  colsample_bytree: 0.8
  gamma: 0.1
  reg_lambda: 1.0
  reg_alpha: 0.1
  seed: 42
  verbosity: 0


#### Train Model

In [6]:
%%time
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=100,
)
int_best_iteration = model.best_iteration
print(f'\nBest iteration: {int_best_iteration}')
print(f'Best validation RMSE: {model.best_score:.4f}')

[0]	train-rmse:1.25228	valid-rmse:1.36539
[100]	train-rmse:0.29879	valid-rmse:1.23863
[117]	train-rmse:0.29274	valid-rmse:1.23718

Best iteration: 18
Best validation RMSE: 1.0620
CPU times: user 1.99 s, sys: 7.23 ms, total: 1.99 s
Wall time: 1.02 s


#### Overfitting Check

In [7]:
arr_pred_train = model.predict(dtrain, iteration_range=(0, int_best_iteration + 1))
arr_pred_valid = model.predict(dvalid, iteration_range=(0, int_best_iteration + 1))
arr_pred_test = model.predict(dtest, iteration_range=(0, int_best_iteration + 1))

flt_rmse_train = np.sqrt(np.mean((y_train[str_target].values - arr_pred_train) ** 2))
flt_rmse_valid = np.sqrt(np.mean((y_valid[str_target].values - arr_pred_valid) ** 2))
flt_rmse_test = np.sqrt(np.mean((y_test[str_target].values - arr_pred_test) ** 2))

df_overfit = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'RMSE (log price)': [round(flt_rmse_train, 4), round(flt_rmse_valid, 4), round(flt_rmse_test, 4)],
})
print(df_overfit.to_string(index=False))

     Split  RMSE (log price)
     Train            0.6223
Validation            1.0620
      Test            1.3182


#### Generate Predictions

In [8]:
# Convert predictions back to actual price scale
arr_price_true = np.expm1(y_test[str_target].values)
arr_price_pred = np.expm1(arr_pred_test)

df_predictions = pd.DataFrame({
    'flt_log_price_true': y_test[str_target].values,
    'flt_log_price_pred': arr_pred_test,
    'flt_price_true': arr_price_true,
    'flt_price_pred': arr_price_pred,
})

print(f'Predictions shape: {df_predictions.shape}')
print(f'\nPredicted price stats:')
print(df_predictions[['flt_price_true', 'flt_price_pred']].describe().round(2))

Predictions shape: (16058, 4)

Predicted price stats:
       flt_price_true  flt_price_pred
count        16058.00        16058.00
mean           633.19          140.28
std          33018.85          183.56
min              0.99           19.97
25%             19.50           43.76
50%             42.99           84.69
75%            125.00          161.05
max        4006600.00         3012.77


#### Save Artifacts

In [9]:
# Save model
model.save_model(f'{str_dirname_output}/xgb_model.json')
print(f'Saved xgb_model.json to {str_dirname_output}')

# Save parameters
with open(f'{str_dirname_output}/params.json', 'w') as f:
    json.dump(dict_params, f, indent=2)
print(f'Saved params.json to {str_dirname_output}')

# Save best iteration
with open(f'{str_dirname_output}/best_iteration.json', 'w') as f:
    json.dump({'best_iteration': int_best_iteration}, f, indent=2)
print(f'Saved best_iteration.json to {str_dirname_output}')

# Save overfitting metrics
dict_overfit = {
    'train_rmse': round(flt_rmse_train, 4),
    'valid_rmse': round(flt_rmse_valid, 4),
    'test_rmse': round(flt_rmse_test, 4),
}
with open(f'{str_dirname_output}/overfitting_metrics.json', 'w') as f:
    json.dump(dict_overfit, f, indent=2)
print(f'Saved overfitting_metrics.json to {str_dirname_output}')

# Save predictions to S3
str_s3_path = f's3://{str_bucket}/{str_task}/test_predictions.csv'
df_predictions.to_csv(str_s3_path, index=False)
print(f'Saved test_predictions.csv to {str_s3_path}')

Saved xgb_model.json to ./output
Saved params.json to ./output
Saved best_iteration.json to ./output
Saved overfitting_metrics.json to ./output
Saved test_predictions.csv to s3://assessment-alt/05_model/test_predictions.csv


#### Takeaways

- **XGBoost regression** trained on log-transformed price with early stopping on validation RMSE
- **Early stopping at 18 rounds** — the model overfits almost immediately. By iteration 100, train RMSE dropped to 0.30 while valid RMSE stayed at 1.24. This suggests the model quickly memorizes card-level price features for seen cards but cannot generalize to the validation period.
- **Train/Valid/Test RMSE**: 0.62 / 1.06 / 1.32 — the increasing gap from valid to test indicates the model struggles with temporal generalization (test data is further in the future)
- **Prediction range is compressed**: model predicts $20–$3K while actual prices range $1–$4M. The log transform helps but extreme outliers are unrecoverable.
- Model artifacts (model JSON, parameters, best iteration) saved locally for reproducibility